### Learning Objectives
- Practice core SQL clauses: SELECT, FROM, WHERE, GROUP BY, HAVING, ORDER BY, LIMIT, DISTINCT.
- Apply JOINs (INNER, LEFT) to combine tables and reason about missing matches.
- Use aggregates (COUNT, SUM, AVG) with grouping and post-group filters.
- Strengthen pattern matching and filtering confidence (LIKE / ILIKE).
- Interpret result sets: sorting, limiting, and handling NULLs with outer joins.

## SQL Practice (Concept + Light Table Use)

Tables available: Customers(customer_id, first_name, last_name, age, country), Orders(order_id, item, amount, customer_id), Shippings(shipping_id, status, customer).

### Quick SQL Command Glance
- SELECT: choose the columns to show in the result set.
- FROM: pick the table (or joined tables) to read from.
- WHERE: filter rows before grouping (row-level filter).
- GROUP BY: roll rows into groups for aggregation (must pair with aggregates).
- HAVING: filter groups after aggregation (group-level filter).
- ORDER BY: sort results; ASC is default, use DESC for reverse.
- LIMIT: cap how many rows return (often paired with ORDER BY).
- DISTINCT: remove duplicate rows from the projection.
- JOIN / INNER JOIN: keep only rows where join keys match across tables.
- LEFT JOIN: keep all left-table rows, even when the right side is missing.
- COUNT / SUM / AVG / MIN / MAX: staple aggregate functions.
- LIKE: pattern matching for text (use % wildcards, case-insensitive in many DBs).

## https://www.programiz.com/sql/online-compiler

## 1) Write a query that shows unique countries from Customers, sorted A→Z.

### Answer
SELECT DISTINCT
  country
FROM Customers
ORDER BY country ASC;

DISTINCT removes repeats, ORDER BY sorts alphabetically.

## 2) Show the top 3 highest order amounts with their order_id (no joins).

### Answer
SELECT
  order_id,
  amount
FROM Orders
ORDER BY amount DESC
LIMIT 3;

ORDER BY DESC ranks biggest first; LIMIT keeps just three rows.

## 3) Using GROUP BY, count how many orders exist per item.

### Answer
SELECT
  item,
  COUNT(*) AS order_count
FROM Orders
GROUP BY item;

GROUP BY buckets rows by item; COUNT tallies each bucket.

## 4) Show only items whose order_count is at least 2 (use HAVING).

### Answer
SELECT
  item,
  COUNT(*) AS order_count
FROM Orders
GROUP BY item
HAVING COUNT(*) >= 2;

HAVING filters after grouping; WHERE cannot see aggregates.

## 5) List customers and their total spend (include those with zero spend).

### Answer
SELECT
  c.first_name,
  c.last_name,
  COALESCE(SUM(o.amount), 0) AS total_spent
FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.first_name, c.last_name;

LEFT JOIN keeps customers without orders; COALESCE turns NULL sums into 0.

## 6) Find customers who have at least one Delivered shipment.

### Answer
SELECT DISTINCT
  c.first_name,
  c.last_name
FROM Customers c
INNER JOIN Shippings s ON c.customer_id = s.customer
WHERE s.status = 'Delivered';

INNER JOIN keeps only matched deliveries; DISTINCT removes duplicates.

## 7) Show the average age of customers per country (sorted by average age descending).

### Answer
SELECT
  country,
  AVG(age) AS avg_age
FROM Customers
GROUP BY country
ORDER BY avg_age DESC;

GROUP BY country, then ORDER BY the computed average age.

## 8) Find order items that start with 'M' (case-insensitive match).

### Answer
SELECT
  order_id,
  item
FROM Orders
WHERE item ILIKE 'M%';

LIKE/ILIKE uses % as a wildcard; this finds items beginning with M.

## 9) Find customers who placed the single highest-value order (include amount and item).

### Answer
SELECT
  c.first_name,
  c.last_name,
  o.item,
  o.amount
FROM Orders o
INNER JOIN Customers c ON o.customer_id = c.customer_id
WHERE o.amount = (SELECT MAX(amount) FROM Orders);

Use a subquery to find the max amount, then join to show who placed that top order.

## 10) Show each country with total orders and total spend (sum of amounts), sorted by spend desc.

### Answer
SELECT
  c.country,
  COUNT(o.order_id) AS total_orders,
  COALESCE(SUM(o.amount), 0) AS total_spend
FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY total_spend DESC;

Aggregates per country using LEFT JOIN to include countries with zero orders; COALESCE avoids NULL totals.

## Question: Data Merging and Lookup Operations with Pandas
## Problem Statement
You are a data analyst at an e-learning platform. You have been given multiple datasets containing student information, course enrollments, and exam scores. Your task is to merge these datasets, handle missing values, perform lookup operations, and generate insights using different join types.

## Task 1: Data Preparation and Missing Value Handling (30 points)
Create the following three DataFrames:

**Students DataFrame:**
```python
students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}
```

**Enrollments DataFrame:**
```python
enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}
```

**Scores DataFrame:**
```python
scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}
```

Tasks:
1. Create all three DataFrames
2. For the students DataFrame:
   - Display null value count and percentage for each column
   - Fill missing 'city' values with 'Unknown'
   - Drop rows where 'name' is missing
3. Display the cleaned students DataFrame

## Task 2: Multiple Join Operations (40 points)
Perform the following join operations and answer questions:

1. **Inner Join**: Merge students and enrollments on student_id
   - How many students appear in the result?
   - Which students from the students table are excluded and why?

2. **Left Join**: Merge students and enrollments on student_id
   - How many total rows are in the result?
   - Which students have null values in course_name and why?

3. **Right Join**: Merge students and enrollments on student_id
   - How many total rows are in the result?
   - Which student_ids appear in the result but don't have student names?

4. **Full Outer Join**: Merge students and enrollments on student_id
   - How many total rows are in the result?
   - Display rows where either student name is null OR course_name is null

5. Add `indicator=True` parameter to the outer join and show the distribution of merge sources (_merge column)

## Task 3: Lookup Operation and Automation (30 points)

1. **Lookup Operation**:
   - Create a dictionary mapping student_id to exam_score from the scores DataFrame
   - Use the `.map()` function to add exam scores to the students DataFrame
   - Display students with their scores (showing NaN for students without scores)

2. **Performance Comparison**:
   - Implement the same score addition using `pd.merge()` with a left join
   - Explain why lookup (map) is more efficient than merge for this scenario

3. **Simple Automation**:
   - Create a function `auto_merge(df1, df2, join_type, key_column)` that:
     - Takes two DataFrames, join type, and key column as input
     - Performs the specified merge
     - Returns a dictionary with: {'result_df': merged_df, 'row_count': count, 'join_type': type}
   - Test your function with at least 2 different join types

## Sample Output
``
=== TASK 1: Data Preparation ===
...
```

## Submission Guidelines
1. Create a Google Colab notebook with clear section headers for each task
2. Use markdown cells to explain your approach and answer all questions
3. Include comments in code explaining each step
4. Ensure all DataFrames are displayed with proper formatting
5. Upload your notebook to Google Drive (set to "Anyone with link can view") OR GitHub (public repository)
6. Submit using this format:
   ```
   Student Name: [Your Name]
   Notebook Link: [Google Drive/GitHub URL]
   All Tasks Completed: Yes/No
   Additional Notes: [Any challenges faced or insights gained]
   ```

#### Task 1 (Text): Data Preparation and Missing Value Handling
Create students, enrollments, and scores DataFrames; show null counts and percentages; fill missing city with 'Unknown'; drop rows with missing name; display cleaned students.

In [ ]:
import pandas as pd
import numpy as np

students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None,
              'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}
enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}
scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

students_df = pd.DataFrame(students_data)
enrollments_df = pd.DataFrame(enrollments_data)
scores_df = pd.DataFrame(scores_data)

# Null value analysis
null_counts = students_df.isnull().sum()
total_rows = len(students_df)
for column in students_df.columns:
    percent = (null_counts[column] / total_rows) * 100
    print(f"Column: {column}, Nulls: {null_counts[column]} ({percent:.2f}%)")

# Handle missing values
students_df['city'].fillna('Unknown', inplace=True)
students_df.dropna(subset=['name'], inplace=True)

print("\nCleaned Students DataFrame:")
print(students_df)

#### Task 2 (Text): Multiple Join Operations
Perform inner, left, right, and full outer joins on student_id; report row counts, excluded/missing students; include indicator-based outer join distribution.

In [ ]:
# Task 2: Multiple Join Operations
# Inner join
inner_result = pd.merge(students_df, enrollments_df, on='student_id', how='inner')
excluded_students = set(students_df['student_id']) - set(inner_result['student_id'])
print(f"Inner rows: {len(inner_result)}")
print(f"Excluded student IDs: {sorted(excluded_students)}")

# Left join
left_result = pd.merge(students_df, enrollments_df, on='student_id', how='left')
null_course_students = left_result[left_result['course_name'].isnull()]['student_id'].tolist()
print(f"Left rows: {len(left_result)}")
print(f"Null course_name students: {null_course_students}")

# Right join
right_result = pd.merge(students_df, enrollments_df, on='student_id', how='right')
no_name_ids = right_result[right_result['name'].isnull()]['student_id'].tolist()
print(f"Right rows: {len(right_result)}")
print(f"Student IDs without names: {no_name_ids}")

# Full outer join
outer_result = pd.merge(students_df, enrollments_df, on='student_id', how='outer')
missing_rows = outer_result[outer_result['name'].isnull() | outer_result['course_name'].isnull()]
print(f"Outer rows: {len(outer_result)}")
print("Rows with missing data:\n", missing_rows[['student_id','name','course_name']])

# Outer join with indicator
outer_with_indicator = pd.merge(students_df, enrollments_df, on='student_id', how='outer', indicator=True)
print("\n_merge distribution:")
print(outer_with_indicator['_merge'].value_counts())

#### Task 3 (Text): Lookup Operation and Automation
Map scores onto students via dictionary/map; compare with merge; create `auto_merge` helper returning result df and counts; test with inner and left joins.

In [ ]:
# Task 3: Lookup Operation and Automation
# Lookup via map
score_dict = dict(zip(scores_df['student_id'], scores_df['exam_score']))
students_with_scores = students_df.copy()
students_with_scores['exam_score'] = students_with_scores['student_id'].map(score_dict)
print("Students with scores (map):\n", students_with_scores)

# Merge comparison
students_merge = pd.merge(students_df, scores_df, on='student_id', how='left')
print("\nStudents with scores (merge):\n", students_merge)
print("Map is O(1) lookups; merge incurs join overhead")

# Automation helper
def auto_merge(df1, df2, join_type, key_column):
    result_df = pd.merge(df1, df2, on=key_column, how=join_type)
    return {'result_df': result_df, 'row_count': len(result_df), 'join_type': join_type, 'df1_rows': len(df1), 'df2_rows': len(df2)}

print("\nTest: inner (students + enrollments)")
print(auto_merge(students_df, enrollments_df, 'inner', 'student_id'))

print("\nTest: left (students + scores)")
print(auto_merge(students_df, scores_df, 'left', 'student_id'))

## Question: Sales Grouping and Aggregations in Pandas
You are a data analyst at an e-commerce company. Your manager needs insights about sales performance across different regions, product categories, and salespersons. Create the synthetic dataset below, then complete all tasks.

```python
import pandas as pd
import random

random.seed(42)

regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']

data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nDataset shape: {df.shape}")
```

### Task 1: Basic Grouping and Single Aggregations
1) Total sales per region using `groupby().sum()` and reset_index
2) Transaction count per product category using `groupby().count()`
3) Average sales per salesperson using `groupby().mean()`
4) Sort regional sales descending to find the top region

### Task 2: Multi-Column Grouping and Multiple Aggregations
1) Group by region and category to sum sales
2) For each salesperson compute sum, mean, count via `agg(['sum','mean','count'])`
3) Sort salesperson metrics by total sales to find top performer
4) Use `.idxmax()` on category totals to find highest-revenue category

### Task 3: Custom Aggregation and Complete Sales Report
1) Define `sales_range = max - min` custom agg
2) Apply sum, mean, min, max, and sales_range by region
3) Regional summary via dict agg: sales_amount sum/mean and customer_id count
4) Explain the multi-level columns produced by dict-based aggregation

### Submission
- Colab notebook `Ecommerce_Sales_Analysis.ipynb`
- Markdown headings per task, run all cells, share link (Drive or GitHub) and test in incognito.

#### Task 1 (Text): Basic Grouping and Single Aggregations
Compute regional sales sums, category transaction counts, average sales per salesperson, and sort regions descending to find the top region.

In [ ]:
import pandas as pd
import random

random.seed(42)
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']
data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}
df = pd.DataFrame(data)

regional_sales = df.groupby('region')['sales_amount'].sum().reset_index()
category_counts = df.groupby('category')['transaction_id'].count().reset_index().rename(columns={'transaction_id': 'transaction_count'})
salesperson_avg = df.groupby('salesperson')['sales_amount'].mean().reset_index().rename(columns={'sales_amount': 'avg_sales'})
regional_sales_sorted = regional_sales.sort_values('sales_amount', ascending=False)

print("Total sales by region:\n", regional_sales)
print("\nTransactions by category:\n", category_counts)
print("\nAverage sales by salesperson:\n", salesperson_avg)
print("\nTop regions (descending):\n", regional_sales_sorted)

#### Task 2 (Text): Multi-Column Grouping and Multiple Aggregations
Sum sales by region-category combos; per salesperson compute sum/mean/count; sort to find top performer; use idxmax to find top category by revenue.

In [ ]:
# Task 2: Multi-Column Grouping and Multiple Aggregations
region_category_sales = df.groupby(['region', 'category'])['sales_amount'].sum().reset_index()
salesperson_metrics = df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()
salesperson_metrics = salesperson_metrics.rename(columns={'sum': 'total_sales', 'mean': 'avg_sales', 'count': 'num_transactions'})
top_salesperson = salesperson_metrics.sort_values('total_sales', ascending=False)
category_sales = df.groupby('category')['sales_amount'].sum()
top_category = category_sales.idxmax()
top_category_amount = category_sales.max()

print("Sales by region & category (sample):\n", region_category_sales.head(10))
print("\nSalesperson metrics:\n", salesperson_metrics)
print("\nTop performer:\n", top_salesperson.head(1))
print(f"\nTop category: {top_category} with ${top_category_amount:,.2f}")

#### Task 3 (Text): Custom Aggregation and Complete Sales Report
Add custom sales range (max-min); aggregate sum/mean/min/max/range by region; build regional summary with sales sum/mean and customer counts; flatten multi-level columns example.

In [ ]:
# Task 3: Custom Aggregation and Complete Sales Report
def sales_range(x):
    return x.max() - x.min()

regional_analysis = df.groupby('region')['sales_amount'].agg(['sum', 'mean', 'min', 'max', sales_range]).reset_index()
regional_analysis = regional_analysis.rename(columns={'sum': 'total_sales', 'mean': 'avg_sales', 'min': 'min_sale', 'max': 'max_sale'})
regional_summary = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean'],
    'customer_id': 'count'
}).reset_index()

print("Regional analysis with custom range:\n", regional_analysis)
print("\nRegional summary (multi-level columns):\n", regional_summary)

# Flatten columns example
regional_summary.columns = ['_'.join(col).strip('_') for col in regional_summary.columns.values]
print("\nFlattened summary columns:\n", regional_summary)

## Question 1: SQL Basics
You are a data analyst at a tech company. The HR department needs insights about employee data. You have access to an `employees` table with columns employee_id, name, department, salary, hire_date. Departments include Engineering, Sales, Marketing, Finance, HR, Operations.

### Task 1: Basic Querying
1. Select all columns for employees in the Marketing department.
2. Display name, department, salary for employees earning more than $90,000.
3. Find employees in either Sales OR Finance.

### Task 2: Sorting and Limiting
1. Top 5 highest-paid employees (salary desc, limit 5).
2. 3 most recently hired Engineering employees (hire_date desc).
3. Oldest employee (by hire_date) who earns > $70,000 and is NOT in HR.

### Task 3: Calculated Fields and Aliases
1. Monthly salary for employees earning > $60,000 (show name, salary, salary/12 as monthly_pay).
2. Finance report: name as employee_name, salary as annual_salary, salary/12 as monthly_salary, sorted by salary desc.

### Submission
- Clear SQL queries and short explanations; optional screenshots from an online SQL editor.

#### Task 1 (Text): Basic Querying
1) All Marketing employees. 2) Name/department/salary for salary > 90000. 3) Employees in Sales or Finance.

**Answer (Text):**
- All Marketing employees:
  SELECT * FROM employees WHERE department = 'Marketing';
- High earners:
  SELECT name, department, salary FROM employees WHERE salary > 90000;
- Sales OR Finance:
  SELECT name, department FROM employees WHERE department = 'Sales' OR department = 'Finance';

#### Task 2 (Text): Sorting and Limiting
1) Top 5 highest-paid (salary desc limit 5). 2) 3 most recent Engineering hires (hire_date desc). 3) Oldest hire earning >70000 not in HR.

**Answer (Text):**
- Top 5 salaries:
  SELECT name, department, salary FROM employees ORDER BY salary DESC LIMIT 5;
- Recent Engineering hires:
  SELECT name, hire_date FROM employees WHERE department = 'Engineering' ORDER BY hire_date DESC LIMIT 3;
- Oldest high earner not HR:
  SELECT name, department, salary, hire_date FROM employees WHERE salary > 70000 AND department != 'HR' ORDER BY hire_date ASC LIMIT 1;

#### Task 3 (Text): Calculated Fields and Aliases
1) Monthly salary for salary > 60000. 2) Finance report with aliases and monthly salary. 3) Extra practice: advanced filtering and department stats.

**Answer (Text):**
- Monthly salary for earners > 60k:
  SELECT name, salary, salary/12 AS monthly_pay FROM employees WHERE salary > 60000;
- Finance report:
  SELECT name AS employee_name, salary AS annual_salary, salary/12 AS monthly_salary FROM employees WHERE department = 'Finance' ORDER BY salary DESC;
- Extra practice: advanced filtering:
  SELECT name, department, salary, hire_date FROM employees WHERE (department = 'Engineering' OR department = 'Sales') AND hire_date < DATE('now', '-3 years') AND salary > 80000;
- Extra practice: department stats:
  SELECT department, COUNT(*) AS employee_count, AVG(salary) AS avg_salary, MIN(salary) AS min_salary, MAX(salary) AS max_salary FROM employees GROUP BY department ORDER BY employee_count DESC;
- Note: DATE('now','-3 years') uses SQLite syntax; adjust date functions for your SQL dialect.